# Split long spreadsheets into chunks of 9999 rows compatible with Alma analytics

## Use case

Sometimes you want to run a report in Alma Analytics, based on a spreadsheet that may contain tens of thousands of lines.

Alma Analytics only supports to filter 9999 values at a time, separated by semicolon.

The code in this notebook will read your large file, take values from the column that you indicate, and split them into groups of 9999 semicolon-separated lines.

Then you can open the `.txt` file and copy-paste these lines into Alma Analytics.
  
**NOTE: The resulting `.txt` file tends to display better using [Notepad++](https://notepad-plus-plus.org/)**

## Step 1 : load the spreadsheet

* The spreadsheet should be loaded into the same folder as the jupyter notebook or Python code.
* Supported formats include `.xlsx` and `.csv`
* It is recommended to use a short, simple file name for the spreadsheet, with no spaces, periods, or special characters.

## Step 2 : Import libraries

This step is primarily getting the Python code ready to read your spreadsheet and generate the output.

All you have to do is run the code!

In [ ]:
try:
    import pandas as pd
    print("pandas imported successfully.")
except ImportError:
    print("pandas not found. Installing...")
    !pip install pandas
    import pandas as pd
    print("pandas installed and imported successfully.")

try:
    import openpyxl
    print("openpyxl imported successfully.")
except ImportError:
    print("openpyxl not found. Installing...")
    !pip install openpyxl
    import openpyxl
    print("openpyxl installed and imported successfully.")


## Step 3 : Indicate the filename and the name of the column with data to be split

3.1 The code will prompt you to enter the filename with the data you want to split.

Make sure you enter the name exactly as it is (case sensitive), including the extension. Example: `journal-info.xlxs`


In [ ]:
df = None
while df is None:
    file_name = input("Please enter the filename (e.g., my_data.csv or my_data.xlsx): ")
    try:
        if file_name.endswith('.csv'):
            df = pd.read_csv(file_name)
            print(f"Successfully loaded CSV file: {file_name}")
        elif file_name.endswith('.xlsx'):
            df = pd.read_excel(file_name)
            print(f"Successfully loaded Excel file: {file_name}")
        else:
            print("Unsupported file format. Please provide a .csv or .xlsx file.")
            continue
    except FileNotFoundError:
        print(f"Error: File '{file_name}' not found. Please verify the filename and extension.")
    except Exception as e:
        print(f"An error occurred while reading the file: {e}")


# Display the first few rows of the DataFrame to confirm it's loaded
if df is not None:
    print("\nFirst 5 rows of the loaded data:")
    display(df.head())

3.2 The code will prompt you to enter the name of the column with the data you want to split into chunks of 9999 items to copy into Alma analytics.

Make sure you enter the name exactly as it is (case sensitive). Example: `MMS ID` will be interpreted differently than `MMS Id` or `MMSID`

In [ ]:
selected_column_data = None
while selected_column_data is None:
    column_name = input("Please enter the name of the column you want to extract (case-sensitive): ")
    try:
        # Check if the DataFrame (df) exists and contains the column
        if df is not None and column_name in df.columns:
            selected_column_data = df[column_name]
            print(f"Successfully selected column: '{column_name}'")
            # Display the first few items of the selected column
            print("\nFirst 5 items of the selected column:")
            display(selected_column_data.head())
        elif df is None:
            print("Error: No DataFrame loaded yet. Please ensure the previous step ran successfully.")
            break # Exit if df is not loaded
        else:
            print(f"Error: Column '{column_name}' not found. Please verify the column name (case-sensitive).")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

## Step 4 : Split the column into chunks and save to plain `.txt` file for copy-pasting into Alma analytics

In [ ]:
import os

# Convert the column to string type and trim spaces
# Ensure selected_column_data exists and is not empty
if selected_column_data is None or selected_column_data.empty:
    print("Error: No column data available for processing. Please ensure the previous steps were completed successfully.")
else:
    processed_data = selected_column_data.astype(str).str.strip()

    # Define the chunk size
    chunk_size = 9999

    # Determine the output filename based on the original file_name
    # Assuming 'file_name' variable exists from the previous cell (e.g., 'my_data.csv')
    if 'file_name' in locals() and file_name:
        # Get the base name without extension
        base_file_name = file_name.rsplit('.', 1)[0] if '.' in file_name else file_name
        output_filename = f"split-{base_file_name}.txt"
    else:
        # Fallback if file_name is not defined (should not happen if previous cells run)
        output_filename = "split-output.txt"
        print("Warning: Original filename not found. Using 'split-output.txt' as fallback.")

    try:
        with open(output_filename, 'w', encoding='utf-8') as f:
            # Iterate through the data in chunks
            for i in range(0, len(processed_data), chunk_size):
                chunk = processed_data.iloc[i : i + chunk_size]
                # Join the values in the chunk with a semicolon
                chunk_line = ";".join(chunk)
                f.write(chunk_line + os.linesep)
                # Add a blank line between groups, but not after the very last group
                if i + chunk_size < len(processed_data):
                    f.write(os.linesep)

        print(f"The data has been split and saved as '{output_filename}'")

    except Exception as e:
        print(f"An error occurred while saving the file: {e}")